In [ ]:
!pip install pandas scikit-learn joblib numpy


CODE STARTS HERE


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

# --- STEP 1: LOAD & CLEAN DATA ---
filename = "KDDTest.arff.csv"

# Read all lines
with open(filename, "r") as f:
    lines = f.readlines()

# Parse the messy ARFF/CSV format
data_lines = []
for line in lines:
    line = line.strip()
    # The data is inside quotes: "0,tcp,..." followed by commas
    # We split by " to extract the content inside the quotes
    parts = line.split('"')

    # Check if we successfully extracted the content (part[1])
    if len(parts) >= 2:
        content = parts[1]
        data_lines.append(content.split(','))

# Define the standard KDD columns
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label"
]

# Create DataFrame
df = pd.DataFrame(data_lines, columns=columns)

# Map KDD columns to System Metrics for the simulation
numerical_cols = ["count", "srv_count", "dst_host_count", "src_bytes", "dst_bytes"]
for col in numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_mapped = df[numerical_cols].copy()
df_mapped.columns = ["cpu", "memory", "disk", "network", "process_count"]
df_mapped["label"] = df["label"]

# Convert labels (0: normal, 1: anomaly)
# We strip any accidental whitespace just in case
df_mapped["label"] = df_mapped["label"].astype(str).str.strip().apply(lambda x: 0 if x == "normal" else 1)

# Filter for normal traffic
normal_df = df_mapped[df_mapped["label"] == 0]
X = normal_df.drop(columns=["label"])

print(f"✅ Data Loaded. Found {len(X)} normal samples for training.")

# --- STEP 2: TRAIN MODEL ---
if len(X) > 0:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
    n_estimators=200,      # Increased from 150 for better stability
    contamination=0.45,    # Increase this! It tells the model to be more suspicious
    max_samples='auto',    # Let the model decide the best sample size
    random_state=42
)

    model.fit(X_scaled)

    joblib.dump(model, "model.pkl")
    joblib.dump(scaler, "scaler.pkl")
    print("✅ Model trained & saved successfully.")
else:
    print("❌ Error: X is still empty. Please check the file content.")

✅ Data Loaded. Found 9711 normal samples for training.
✅ Model trained & saved successfully.


In [ ]:
import numpy as np
import time
from collections import deque
import joblib

# Load the trained model
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")

WINDOW_SIZE = 10
window = deque(maxlen=WINDOW_SIZE)

def explain(features):
    """Generates a human-readable reason for the anomaly based on thresholds."""
    cpu, mem, disk, net, proc = features
    reasons = []

    # Thresholds tuned for the simulation values
    if cpu > 80: reasons.append("High CPU usage")
    if proc > 300: reasons.append("Process explosion")
    if net > 50000: reasons.append("Network spike")

    return reasons if reasons else ["Unknown anomaly pattern"]

def detect(stream):
    print("\nStarting Stream Detection...")
    for sample in stream:
        window.append(sample)

        if len(window) == WINDOW_SIZE:
            # Calculate average behavior over the window
            avg = np.mean(window, axis=0).reshape(1, -1)

            # Scale input to match training data distribution
            scaled = scaler.transform(avg)

            # Predict
            pred = model.predict(scaled)
            score = abs(model.decision_function(scaled)[0])

            if pred[0] == -1:
                print(f"🚨 ANOMALY DETECTED | Risk Score: {round(score,3)}")
                print("   Reason:", ", ".join(explain(avg[0])))
            else:
                # Optional: Print dot for normal behavior to reduce clutter
                print(".", end="", flush=True)

        time.sleep(0.1) # Faster sleep for demo purposes
    print("\nStream finished.\n")

In [ ]:
import warnings
# Filter out the specific feature name warning
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [ ]:
import numpy as np
import time
import random
from collections import deque
import joblib
import warnings

# Suppress warnings for clean output
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Load the 5-feature model and scaler
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")

WINDOW_SIZE = 10
window = deque(maxlen=WINDOW_SIZE)

def explain(features):
    """Generates a human-readable reason for the anomaly."""
    cpu, mem, disk, net, proc = features
    reasons = []
    if cpu > 80: reasons.append("High CPU usage")
    if proc > 2000: reasons.append("Process explosion")
    if net > 20000: reasons.append("Network spike")
    return reasons if reasons else ["Unknown anomaly pattern"]

def generate_live_stream(mode="normal", steps=20):
    """Generates 5-feature data with a 'Normal' lead-in."""
    stream = []
    print(f"\n--- 🔄 Simulating '{mode}' sequence ---")

    for i in range(steps):
        # Force the lead-in to be normal so the sliding window fills with safe data first
        if mode == "normal" or i < 10:
            row = [
                random.randint(10, 40),   # CPU
                random.randint(20, 50),   # Mem
                random.randint(30, 60),   # Disk
                random.randint(500, 2000),# Net
                random.randint(50, 150)   # Proc
            ]
        elif mode == "cpu_spike":
            row = [random.randint(600, 800), random.randint(40, 60), random.randint(50, 70),
                   random.randint(1000, 3000), random.randint(100, 200)]
        elif mode == "network_attack":
            row = [random.randint(30, 50), random.randint(40, 60), random.randint(50, 70),
                   random.randint(60000, 90000), random.randint(100, 200)]

        stream.append(row)
    return stream

def detect(stream):
    print("Initializing Detection Engine...")
    for sample in stream:
        window.append(sample)

        if len(window) == WINDOW_SIZE:
            # 1. Calculate window average (5 features)
            avg = np.mean(window, axis=0).reshape(1, -1)

            # 2. Scale input (Expects exactly 5 features)
            scaled = scaler.transform(avg)

            # 3. AI Inference
            pred = model.predict(scaled)
            score = abs(model.decision_function(scaled)[0])

            if pred[0] == -1:
                print(f"🚨 ALERT! Risk Score: {round(score,3)} | Reason: {', '.join(explain(avg[0]))}")
            else:
                # Prove the 'Normal' lead-in is working by showing healthy CPU status
                print(f"✅ System Healthy (CPU: {int(sample[0])}%)")

        time.sleep(0.1)
    print("Stream sequence completed.")

# --- EXECUTION ---
detect(generate_live_stream("normal"))
detect(generate_live_stream("cpu_spike"))


--- 🔄 Simulating 'normal' sequence ---
Initializing Detection Engine...
🚨 ALERT! Risk Score: 0.083 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.089 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.085 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.084 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.084 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.081 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.08 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.085 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.088 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.083 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.085 | Reason: Unknown anomaly pattern
Stream sequence completed.

--- 🔄 Simulating 'cpu_spike' sequence ---
Initializing Detection Engine...
🚨 ALERT! Risk Score: 0.082 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.08 | Reason: Unknown anomaly pattern
🚨 ALERT! Risk Score: 0.081 | Reas

In [ ]:


# Suppress the messy warnings to make output clean
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Load the trained model
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")

WINDOW_SIZE = 10
window = deque(maxlen=WINDOW_SIZE)

def explain(features):
    """Generates a human-readable reason for the anomaly based on thresholds."""
    cpu, mem, disk, net, proc = features
    reasons = []

    # Thresholds tuned for the simulation values
    if cpu > 80: reasons.append("High CPU usage")
    if proc > 2000: reasons.append("Process explosion") # Tuned threshold
    if net > 20000: reasons.append("Network spike")

    return reasons if reasons else ["Unknown anomaly pattern"]

def generate_live_stream(mode="normal", steps=15):
    """Generates random data based on the mode."""
    stream = []
    print(f"\n--- 🔄 Generating '{mode}' traffic stream ---")

    for _ in range(steps):
        if mode == "normal":
            # Normal: All values low and safe
            row = [
                random.randint(10, 40),   # CPU
                random.randint(20, 50),   # Mem
                random.randint(30, 60),   # Disk
                random.randint(500, 2000),# Net
                random.randint(50, 150)   # Proc
            ]
        elif mode == "cpu_spike":
            # CPU forced to 600 to exceed Dataset Max (511)
            row = [
                random.randint(600, 800), # CPU: CRITICAL SPIKE
                random.randint(40, 60),
                random.randint(50, 70),
                random.randint(1000, 3000),
                random.randint(100, 200)
            ]
        elif mode == "network_attack":
            # Network forced to 60k+ to exceed Dataset Max (~12k)
            row = [
                random.randint(30, 50),
                random.randint(40, 60),
                random.randint(50, 70),
                random.randint(60000, 90000), # Net: HUGE SPIKE
                random.randint(100, 200)
            ]
        elif mode == "process_bomb":
            # Proc forced to 100k+ to act as an outlier
            row = [
                random.randint(20, 40),
                random.randint(80, 95),   # High Mem
                random.randint(50, 70),
                random.randint(1000, 5000),
                random.randint(50000, 100000) # Proc: HUGE SPIKE
            ]

        stream.append(row)
    return stream

def detect(stream):
    print("Starting detection...")
    for sample in stream:
        window.append(sample)

        if len(window) == WINDOW_SIZE:
            # Calculate average behavior over the window
            avg = np.mean(window, axis=0).reshape(1, -1)

            # Scale input
            scaled = scaler.transform(avg)

            # Predict
            pred = model.predict(scaled)
            score = abs(model.decision_function(scaled)[0])

            if pred[0] == -1:
                print(f"🚨 ALERT! Risk Score: {round(score,3)}")
                # Show the relevant metric that caused the spike
                if avg[0][0] > 100:
                    print(f"   Values: CPU={int(avg[0][0])}% (Abnormal)")
                elif avg[0][3] > 20000:
                    print(f"   Values: Net={int(avg[0][3])} bytes (Abnormal)")
                elif avg[0][4] > 2000:
                    print(f"   Values: Procs={int(avg[0][4])} (Abnormal)")

                print(f"   Reason: {', '.join(explain(avg[0]))}")
            else:
                print(f"✅ Normal (CPU: {sample[0]}%)")

        time.sleep(0.1)
    print("Stream finished.\n")

# --- RUN THE SIMULATION (ALL 4 SCENARIOS) ---

# 1. Normal
detect(generate_live_stream("normal"))

# 2. CPU Spike (DoS Attack)
detect(generate_live_stream("cpu_spike"))

# 3. Process Explosion (Fork Bomb)
detect(generate_live_stream("process_bomb"))

# 4. Network Spike (DDoS)
detect(generate_live_stream("network_attack"))